# 扩展节点：个人账户与家庭共享

这一节点让同一个应用中的不同账户拥有独立个人物品，也能通过家庭码加入同一家庭并共享家庭物品。

## 1. 本节点目标

解决三个生活化问题：个人物品不能被其他账户看到；一个账户可以属于多个家庭；同一家人在家庭空间内看到和修改同一份物品数据。

## 2. 完成结果与验收

- 支持邮箱或手机号注册、登录，密码不以明文保存。
- 每个账户拥有独立个人空间。
- 用户可以创建多个家庭并获得 8 位家庭码。
- 第二个账户输入家庭码后可看到第一个账户添加的家庭物品。
- 个人空间、不同家庭之间的数据相互隔离。
- 账户会记住手动选择的城市，刷新或重新登录后无需重选。
- 尚未保存城市的手机可在浏览器授权后自动定位，精确坐标不写入数据库。
- 物品状态页每页最多 4 件，支持左右翻页。
- 页面只使用衣物、床上用品、玩偶、宠物用品四种分类颜色。
- 浏览器端到端流程和全量 64 项测试通过。

## 3. 本节点文件结构

```text
src/smart_laundry/accounts.py      注册、登录、家庭码和成员关系
src/smart_laundry/database.py      账户/家庭表与物品范围迁移
src/smart_laundry/repositories.py  个人或家庭作用域查询
src/smart_laundry/models.py        四种通俗物品分类
app.py                            登录、空间切换、家庭管理和分页
tests/test_accounts.py             账户、家庭共享和隔离测试
docs/resume.md                     可验证的简历项目描述
```

## 4. 关键代码解释

`AccountRepository.register()` 使用随机盐和 PBKDF2 计算密码摘要，数据库只保存盐和摘要。`update_preferred_city()` 把常住城市写入账户，使页面重载后仍能恢复。手机定位组件只在账户没有城市时请求一次浏览器授权，再把坐标转换为城市；数据库不保存经纬度。`create_family()` 生成不易混淆的 8 位家庭码，并把创建者写入成员表。`ItemRepository` 接收 `owner_user_id` 或 `family_id`：个人空间查询要求物品属于当前账户且没有家庭 ID；家庭空间只查询当前家庭 ID。Agent、编辑和活动记录继续使用同一个已限定范围的 repository，因此不会绕过空间隔离。

In [ ]:
def scope_label(owner_user_id=None, family_id=None):
    if family_id is not None:
        return f'家庭空间 {family_id}'
    if owner_user_id is not None:
        return f'个人空间 {owner_user_id}'
    return '测试或迁移使用的未限定范围'

print(scope_label(owner_user_id=7))
print(scope_label(family_id=3))

## 5. 数据流

```mermaid
flowchart LR
 A[邮箱或手机号注册] --> B[PBKDF2 密码摘要]
 B --> C[个人空间]
 C --> D{选择空间}
 D -->|个人| E[按 owner_user_id 查询]
 D -->|创建家庭| F[生成家庭码]
 D -->|输入家庭码| G[写入家庭成员关系]
 F --> H[家庭空间]
 G --> H
 H --> I[按 family_id 共享物品]
 I --> J[家庭成员共同更新状态]
```

## 6. 关键概念

- **身份认证**：确认登录者是否知道该账户密码。
- **密码哈希**：把密码转换成不可直接还原的摘要。
- **盐**：每个账户不同的随机数据，防止相同密码产生相同摘要。
- **成员关系表**：记录哪个用户属于哪个家庭。
- **作用域查询**：每次读写都自动带上个人或家庭条件。
- **共享数据库**：多个会话访问同一份服务端数据。

## 7. 为什么这样设计

首版使用 SQLite 和应用内账户，而没有立刻引入 OAuth、短信验证码和云数据库，可以用最少依赖验证多账户与家庭共享的业务模型。代价是本地启动时只有当前电脑能访问；要实现真正跨设备同步，需要把应用和数据库部署到共享服务器，并进一步增加 HTTPS、会话持久化、找回密码和访问审计。

## 8. 常见错误与排查

1. **注册后看不到旧物品**：只有第一个账户会接管升级前的本地物品。
2. **家庭码无效**：检查是否完整输入 8 位字符，系统会自动忽略大小写。
3. **加入家庭后仍在个人空间**：在顶部空间选择器中切换家庭。
4. **另一台电脑无法打开 localhost**：localhost 只代表当前电脑，需要部署公网服务。
5. **登录后刷新退出**：当前会话保存在 Streamlit session，首版没有长期登录 Cookie。
6. **物品串到其他家庭**：检查 repository 是否用当前 family_id 创建。

## 9. 面试可能追问

**问：怎样避免家庭之间数据串读？** 答：UI 不直接拼 SQL，所有物品读写都通过带 owner_user_id 或 family_id 的 repository，测试验证家庭物品不能从个人 repository 读取。

**问：为什么不用明文密码？** 答：明文泄露会直接暴露账户，项目使用随机盐和 PBKDF2 多轮计算摘要，并用恒定时间比较验证。

**问：这算真正实时同步吗？** 答：业务共享模型已完成，同一服务数据库中的会话会读取相同数据；尚未部署公网和 WebSocket，因此不能声称已有生产级跨设备实时推送。

**追问：生产环境会怎么升级？** 答：使用托管 PostgreSQL、成熟身份服务、HTTPS、安全 Cookie、审计日志和数据库级权限，并根据需要增加消息推送或轮询。

## 10. 必须掌握的最少知识

账户 ID 标识个人，家庭 ID 标识共享空间；家庭成员表连接两者；物品只能属于个人或一个家庭；密码只保存哈希；本地共享逻辑和公网部署不是同一件事。

## 11. 可自测小题

1. 为什么相同密码也要使用不同的盐？
2. 家庭码本身是否等于登录密码？
3. 个人物品和家庭物品分别使用什么条件查询？
4. 为什么 localhost 不能放在简历里当在线 Demo？
5. Agent 为什么也能遵守家庭数据范围？

<details><summary>参考答案</summary>

1. 防止相同密码产生相同摘要并降低预计算攻击风险。2. 不是；家庭码只用于建立成员关系，登录仍需要个人密码。3. owner_user_id 和 family_id。4. 它只能从当前电脑访问。5. Agent 工具接收的就是已限定范围的 repository。

</details>

## 12. 动手小练习

1. 创建两个测试账户，让第二个账户用家庭码加入第一个账户的家庭。
2. 在家庭空间添加一件物品，再切换个人空间比较列表差异。
3. 运行 `python -m pytest tests/test_accounts.py`，观察共享与隔离测试。

## 13. 本节点术语表

| 术语 | 简单解释 |
|---|---|
| authentication | 验证登录者身份 |
| PBKDF2 | 多轮计算密码摘要的标准方法 |
| salt | 每个账户单独生成的随机值 |
| membership | 用户与家庭之间的成员关系 |
| scope | 当前操作允许访问的数据范围 |
| deployment | 把本地应用放到可被其他设备访问的服务器 |

## 14. 下一节点连接

下一步可以把当前 SQLite 共享模型迁移到托管 PostgreSQL，并加入正式会话、HTTPS 和公网部署；完成后再把 GitHub 与在线 Demo 链接补入简历。